# Proyecto Integrador – Hito 1
## Construcción del Dataset Analítico – SuperTienda
Alessandro Escalante




**Objetivo del hito:** Explorar la base de datos SuperTienda, identificar relaciones entre tablas y construir un dataset analítico consolidado para análisis posteriores.

In [ ]:
# Descarga de database desde drive automatico

import os
import re
import sqlite3
import requests
import pandas as pd

DB_FILE = 'SuperTienda_Espanol.db'
DRIVE_FILE_ID = '1ztSBwhT10K8ZH5GAgiwpQ9nC0MkMmFsW'

def descargar_desde_drive(file_id, output_path):
    url = f'https://drive.google.com/uc?export=download&id={file_id}'
    session = requests.Session()
    response = session.get(url, stream=True)
    match = re.search(r'confirm=([0-9A-Za-z_-]+)', response.text)
    if match:
        response = session.get(f'{url}&confirm={match.group(1)}', stream=True)
    response.raise_for_status()
    with open(output_path, 'wb') as f:
        f.write(response.content)

if not os.path.exists(DB_FILE):
    descargar_desde_drive(DRIVE_FILE_ID, DB_FILE)


## Fase 1 – Exploración de la base de datos

En esta primera etapa se analiza la estructura de la base de datos SuperTienda. Para ello, se identifican las tablas disponibles, las columnas de cada una y las relaciones posibles entre ellas. El objetivo es seleccionar la información relevante para construir un dataset analítico útil para etapas posteriores del proyecto.

In [ ]:
# Conexión a la base de datos
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'
%sql sqlite:///SuperTienda_Espanol.db


In [ ]:
conn = sqlite3.connect("SuperTienda_Espanol.db")

In [ ]:
# Identificamos las tablas disponibles que tenemos en la base de datos
%%sql tablas <<
SELECT name
FROM sqlite_master
WHERE type = 'table';

 * sqlite:///SuperTienda_Espanol.db
Done.
Returning data to local variable tablas


La base de datos contiene 8 tablas: `Clientes`, `Productos`, `Geografia`, `Cliente_Ubicacion`, `Campanias`, `Pedidos`, `Detalle_Pedido`, `Soporte`

In [ ]:
# Visualizamos las tablas
print(tablas)

+-------------------+
|        name       |
+-------------------+
|      Clientes     |
|     Productos     |
|     Geografia     |
| Cliente_Ubicacion |
|     Campanias     |
|      Pedidos      |
|   Detalle_Pedido  |
|      Soporte      |
+-------------------+


Exploramos las columnas de cada tabla importante para nuestro enfoque en VENTAS

In [ ]:
%%sql
SELECT *
FROM Clientes
LIMIT 3;

 * sqlite:///SuperTienda_Espanol.db
Done.


ID_Cliente,Nombre_Cliente,Segmento,Fecha_Registro,Canal_Preferido,Puntaje_Fidelidad,Codigo_Interno
C0001,Fernanda Díaz,Consumidor,2023-02-18,Web,55,CLI-6635-Z
C0002,Valentina Muñoz,Home Office,2023-07-30,Web,58,CLI-2291-Z
C0003,Diego Castillo,Corporativo,2024-03-28,App,100,CLI-2139-X


In [ ]:
%%sql
SELECT *
FROM Campanias
LIMIT 3;

 * sqlite:///SuperTienda_Espanol.db
Done.


ID_Campania,Nombre_Campania,Canal_Marketing,Presupuesto,Temporada,Responsable,Objetivo
MKT001,Campaña Cliente VIP 1,Email,1975985.03,Invierno,Equipo CRM,Conversión
MKT002,Campaña Ahorro 2,Afiliados,554708.22,Navidad,Equipo CRM,Reconocimiento
MKT003,Campaña Activa 3,Afiliados,445067.11,Verano,None,Reconocimiento


In [ ]:
%%sql
SELECT *
FROM Productos
LIMIT 3;

 * sqlite:///SuperTienda_Espanol.db
Done.


ID_Producto,Nombre_Producto,Categoria,Subcategoria,Precio_Lista,Costo_Unitario,Marca,Activo,Codigo_Barra
P0001,Accesorios Flex 1,Tecnología,Accesorios,445.84,210.5,Alerce,1,7154379937124
P0002,Accesorios Smart 2,Tecnología,Accesorios,10.06,5.92,Vector,1,5756924593752
P0003,Accesorios Plus 3,Tecnología,Accesorios,868.26,685.1,Altura,1,3564651939828


In [ ]:
%%sql
SELECT *
FROM Pedidos
LIMIT 3;

 * sqlite:///SuperTienda_Espanol.db
Done.


ID_Pedido,ID_Cliente,Fecha_Pedido,Fecha_Envio,Modo_Envio,ID_Campania,Estado_Pedido,Observacion_Interna
ORD-2023-00001,C0174,2025-03-31,2025-04-04,Estándar,None,Completado,Cliente frecuente
ORD-2023-00002,C0254,2024-03-25,2024-03-25,Segunda Clase,MKT005,Completado,Sin observaciones
ORD-2023-00003,C0203,2024-02-20,2024-02-24,Estándar,MKT001,Completado,Revisar descuento aplicado


In [ ]:
%%sql
SELECT *
FROM Detalle_Pedido
LIMIT 3;

 * sqlite:///SuperTienda_Espanol.db
Done.


ID_Detalle,ID_Pedido,ID_Producto,Cantidad,Descuento,Ventas,Ganancia,Prioridad
1,ORD-2023-00001,P0082,1,0.0,590.86,322.42,Crítica
2,ORD-2023-00001,P0119,7,0.0,4341.68,1907.85,Media
3,ORD-2023-00001,P0048,2,0.0,1250.86,281.32,Crítica


In [ ]:
%%sql
SELECT *
FROM Geografia
LIMIT 3;

 * sqlite:///SuperTienda_Espanol.db
Done.


ID_Ubicacion,Pais,Region,Estado,Ciudad,Codigo_Postal,Zona_Logistica,Latitud_Ref
UB001,Chile,Norte,Antofagasta,Antofagasta,1240000,A,-52.1246
UB002,Chile,Norte,Coquimbo,La Serena,1700000,C,-44.4288
UB003,Chile,Norte,Tarapacá,Iquique,1100000,B,-27.2235


Descartamos `Cliente_Ubicacion`, `Soporte` y porque no aportan informacion relevante para el analisis de ventas

#Fase 2: Extracción de datos (SQL y Pandas)
Debes construir una consulta SQL que integre información relevante y convertir esta consulta en un DataFrame de Pandas.

Para ayudarte a comenzar, puedes apoyarte en uno de los siguientes enfoques:

Opción 1: Enfoque de Ventas: Integra información relacionada con pedidos, detalle de pedidos y productos para analizar ventas, cantidades y categorías.

Construimos una consulta SQL que integra las tablas seleccionadas.

In [ ]:
query_dataset = """
SELECT
  Pe.ID_Pedido,
  Pe.ID_Cliente,
  date(Pe.Fecha_Pedido) AS Fecha_Pedido,
  date(Pe.Fecha_Envio) AS Fecha_Envio,
  DP.ID_Detalle,
  DP.ID_Producto,
  P.Nombre_Producto,
  P.Categoria,
  P.Subcategoria,
  P.Marca,
  DP.Cantidad,
  DP.Ventas,
  DP.Ganancia,
  P.Precio_Lista,
  P.Costo_Unitario
FROM Pedidos AS Pe
INNER JOIN Detalle_Pedido AS DP
  ON Pe.ID_Pedido = DP.ID_Pedido
INNER JOIN Productos AS P
  ON DP.ID_Producto = P.ID_Producto;
"""

df_analitico = pd.read_sql_query(query_dataset, conn)

print(f"Filas extraidas: {df_analitico.shape[0]:,}".replace(",", "."))
print(f"Columnas extraidas: {df_analitico.shape[1]}")
display(df_analitico.head())

Filas extraidas: 3.026
Columnas extraidas: 15


,ID_Pedido,ID_Cliente,Fecha_Pedido,Fecha_Envio,ID_Detalle,ID_Producto,Nombre_Producto,Categoria,Subcategoria,Marca,Cantidad,Ventas,Ganancia,Precio_Lista,Costo_Unitario
0,ORD-2023-00001,C0174,2025-03-31,2025-04-04,1,P0082,Papelería Eco 2,Oficina,Papelería,Orion,1,590.86,322.42,590.86,268.44
1,ORD-2023-00001,C0174,2025-03-31,2025-04-04,2,P0119,Sobres Plus 9,Oficina,Sobres,Andes,7,4341.68,1907.85,620.24,347.69
2,ORD-2023-00001,C0174,2025-03-31,2025-04-04,3,P0048,Sillas Plus 8,Muebles,Sillas,Orion,2,1250.86,281.32,625.43,484.77
3,ORD-2023-00001,C0174,2025-03-31,2025-04-04,4,P0079,Almacenamiento Nova 9,Muebles,Almacenamiento,Alerce,5,244.35,64.85,48.87,35.90
4,ORD-2023-00002,C0254,2024-03-25,2024-03-25,5,P0087,Papelería Flex 7,Oficina,Papelería,Andes,1,629.31,115.33,740.37,525.57


In [ ]:
# ¿Que categoria vende mas y cuanto se vende por categoria?
%%sql
SELECT
P.Nombre_Producto,
P.Categoria,
SUM(DP.Cantidad) AS Cantidad
FROM Productos AS P
INNER JOIN Detalle_Pedido AS DP
ON P.ID_Producto = DP.ID_Producto
GROUP BY P.Categoria
ORDER BY Cantidad DESC;


 * sqlite:///SuperTienda_Espanol.db
Done.


Nombre_Producto,Categoria,Cantidad
Computadores Plus 1,Tecnología,4834
Sillas Plus 8,Muebles,4466
Papelería Eco 2,Oficina,4248


In [ ]:
# Ventas por mes
%%sql
SELECT
  strftime('%Y-%m',Pe.Fecha_pedido) AS Mes,
  ROUND(SUM(DP.ventas),2)AS Ventas_totales
  FROM Detalle_Pedido AS DP
JOIN  Pedidos AS Pe
  ON Pe.ID_Pedido = DP.ID_Pedido
GROUP BY Mes
ORDER BY Mes DESC;

 * sqlite:///SuperTienda_Espanol.db
Done.


Mes,Ventas_totales
2025-04,189506.6
2025-03,178855.29
2025-02,171214.85
2025-01,235955.67
2024-12,179807.58
2024-11,205788.57
2024-10,208616.55
2024-09,216590.54
2024-08,171325.59
2024-07,177143.49


In [ ]:
# Marcas mas vendidas
%%sql
SELECT
  P.Marca AS Marcas,
  P.Categoria AS Categoria,
  P.Subcategoria AS Subcategoria,
  COUNT(DP.Cantidad) AS Cantidad_Productos
FROM Productos AS P
INNER JOIN Detalle_Pedido AS DP
  ON P.ID_Producto = DP.ID_Producto
GROUP BY P.Marca
ORDER BY Cantidad DESC;

 * sqlite:///SuperTienda_Espanol.db
Done.


Marcas,Categoria,Subcategoria,Cantidad_Productos
Andes,Oficina,Sobres,452
Vector,Oficina,Archivadores,456
Altura,Oficina,Archivadores,347
Alerce,Muebles,Almacenamiento,368
Patagonia,Tecnología,Computadores,452
Nova,Muebles,Libreros,141
Lumen,Oficina,Etiquetas,363
Orion,Oficina,Papelería,447


In [ ]:
# Los 5 Productos mas vendidos
%%sql
SELECT P.Nombre_Producto as "Productos mas vendidos",
SUM(DP.Cantidad) AS "Ventas totales"
FROM Productos AS P
INNER JOIN Detalle_Pedido AS DP
ON P.ID_Producto = DP.ID_Producto

GROUP BY P.ID_Producto, P.Nombre_Producto
ORDER BY "Ventas totales" DESC
LIMIT 5;

 * sqlite:///SuperTienda_Espanol.db
Done.


Productos mas vendidos,Ventas totales
Almacenamiento Eco 6,187
Almacenamiento Smart 4,175
Periféricos Eco 2,167
Sobres Flex 1,156
Archivadores Nova 2,150


## Fase 3 - Limpieza y transformacion con Pandas

En esta fase se prepara el dataset para que sea consistente y facil de usar en visualizaciones posteriores. Se corrigen tipos de datos, se revisan nulos y duplicados, se normalizan campos de texto y se crean variables calculadas para enriquecer el analisis.


In [ ]:
import pandas as pd

df_limpio = df_analitico.copy()

# 1) Definición y conversión de tipos de datos
columnas_fecha = ["Fecha_Pedido", "Fecha_Envio"]
columnas_numericas = ["Cantidad", "Ventas", "Ganancia", "Precio_Lista", "Costo_Unitario"]
columnas_texto = ["Nombre_Producto", "Categoria", "Subcategoria", "Marca"]

for columna in columnas_fecha:
    df_limpio[columna] = pd.to_datetime(df_limpio[columna], errors="coerce")

for columna in columnas_numericas:
    df_limpio[columna] = pd.to_numeric(df_limpio[columna], errors="coerce")

# Convertir las columnas de texto a tipo 'string' y eliminar espacios en blanco
for columna in columnas_texto:
    df_limpio[columna] = df_limpio[columna].astype("string").str.strip()

# 2) Diagnóstico de calidad antes del filtrado
print("Valores nulos por columna antes de limpiar:")
display(df_limpio.isna().sum().to_frame("nulos"))
print(f"Filas duplicadas exactas: {df_limpio.duplicated().sum()}")

# 3) Tratamiento de datos inválidos y duplicados para el enfoque comercial
filas_antes = len(df_limpio)
df_limpio = df_limpio.drop_duplicates().copy() # Eliminar filas duplicadas exactas

# Eliminar filas con valores nulos en columnas críticas
df_limpio = df_limpio.dropna(subset=["ID_Pedido", "ID_Producto", "Fecha_Pedido", "Nombre_Producto", "Categoria", "Cantidad", "Ventas"]).copy()

# Filtrar datos por condiciones de negocio (ej. cantidad positiva, ventas no negativas)
df_limpio = df_limpio[(df_limpio["Cantidad"] > 0) & (df_limpio["Ventas"] >= 0)].copy()
filas_despues = len(df_limpio)

# 4) Variables calculadas relevantes
df_limpio["Mes"] = df_limpio["Fecha_Pedido"].dt.to_period("M").astype(str)
df_limpio["Año"] = df_limpio["Fecha_Pedido"].dt.year
df_limpio["Dias_Envio"] = (df_limpio["Fecha_Envio"] - df_limpio["Fecha_Pedido"]).dt.days
df_limpio["Venta_Promedio_Unidad"] = df_limpio["Ventas"] / df_limpio["Cantidad"]
df_limpio["Costo_Total_Estimado"] = df_limpio["Costo_Unitario"] * df_limpio["Cantidad"]

print(f"\nFilas antes de limpiar: {filas_antes:,}".replace(",", "."))
print(f"Filas después de limpiar: {filas_despues:,}".replace(",", "."))
print(f"Columnas finales: {df_limpio.shape[1]}")
print("Tipos de datos finales:")
display(df_limpio.dtypes.to_frame("tipo"))
print("\nFilas del DataFrame limpio:")
display(df_limpio.head())

Valores nulos por columna antes de limpiar:


,nulos
ID_Pedido,0
ID_Cliente,0
Fecha_Pedido,0
Fecha_Envio,0
ID_Detalle,0
ID_Producto,0
Nombre_Producto,0
Categoria,0
Subcategoria,0
Marca,0


Filas duplicadas exactas: 0

Filas antes de limpiar: 3.026
Filas después de limpiar: 3.026
Columnas finales: 20
Tipos de datos finales:


,tipo
ID_Pedido,object
ID_Cliente,object
Fecha_Pedido,datetime64[ns]
Fecha_Envio,datetime64[ns]
ID_Detalle,int64
ID_Producto,object
Nombre_Producto,string[python]
Categoria,string[python]
Subcategoria,string[python]
Marca,string[python]



Filas del DataFrame limpio:


,ID_Pedido,ID_Cliente,Fecha_Pedido,Fecha_Envio,ID_Detalle,ID_Producto,Nombre_Producto,Categoria,Subcategoria,Marca,Cantidad,Ventas,Ganancia,Precio_Lista,Costo_Unitario,Mes,Año,Dias_Envio,Venta_Promedio_Unidad,Costo_Total_Estimado
0,ORD-2023-00001,C0174,2025-03-31,2025-04-04,1,P0082,Papelería Eco 2,Oficina,Papelería,Orion,1,590.86,322.42,590.86,268.44,2025-03,2025,4,590.86,268.44
1,ORD-2023-00001,C0174,2025-03-31,2025-04-04,2,P0119,Sobres Plus 9,Oficina,Sobres,Andes,7,4341.68,1907.85,620.24,347.69,2025-03,2025,4,620.24,2433.83
2,ORD-2023-00001,C0174,2025-03-31,2025-04-04,3,P0048,Sillas Plus 8,Muebles,Sillas,Orion,2,1250.86,281.32,625.43,484.77,2025-03,2025,4,625.43,969.54
3,ORD-2023-00001,C0174,2025-03-31,2025-04-04,4,P0079,Almacenamiento Nova 9,Muebles,Almacenamiento,Alerce,5,244.35,64.85,48.87,35.90,2025-03,2025,4,48.87,179.50
4,ORD-2023-00002,C0254,2024-03-25,2024-03-25,5,P0087,Papelería Flex 7,Oficina,Papelería,Andes,1,629.31,115.33,740.37,525.57,2024-03,2024,0,629.31,525.57


## Fase 4 - Exportacion del dataset


El archivo DataSet_Limpio.cs es el entregable final del hito junto con este notebook. Queda listo para descargarse desde colab y para usarse en powrbi u otra herramienta de visualizacion

In [ ]:
#Exportacion de datos en csv limpio

OUTPUT_CSV = "DataSet_Limpio.csv"

df_limpio.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Archivo exportado: {OUTPUT_CSV}")

try:
    from google.colab import files
    files.download(OUTPUT_CSV)
except ModuleNotFoundError:
    print("Descarga automatica disponible solo en Google Colab")


Archivo exportado: DataSet_Limpio.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Reflexión

### 🗂️ ¿Qué tablas se utilizaron y por qué?

Se utilizaron principalmente las tablas relacionadas como el **Nombre_Producto, Ventas y Ganancia**, ya que contienen la información clave para analizar la preferencia de los clientes y los productos más demandados.

Estas tablas permiten reconstruir cada venta que facilita el análisis del rendimiento comercial y la toma de decisiones basada en datos.

---

### 🧾 ¿Qué columnas se seleccionaron y cuáles se descartaron?

Se seleccionaron las columnas más relevantes para el análisis:

* ID_Pedido
* ID_Producto
* Nombre_Producto
* Categoría
* Fecha_Pedido
* Cantidad
* Ventas
* Costo_Unitario
* Modo_Envío
* Estado_Pedido

Se descartaron aquellas columnas que no aportaban valor analítico directo o que eran redundantes, como identificadores innecesarios o variables sin impacto en el análisis de ventas.

---

### 📈 ¿Qué tipo de análisis permitirá este dataset?

Este dataset permitirá realizar análisis descriptivos , tales como:

* Evolución de las ventas por mes
* Identificación de los productos más vendidos y más rentables
* Análisis del comportamiento de compra de los clientes
* Evaluación de ganancias y costos por producto

